# Lançamento de um experimento (nó de login do Apuana)

Este notebook **não treina nada**. Ele:
1. confere segredos, dados e o envio ao GitHub **antes** de gastar GPU;
2. transforma o experimento (`configs/experimentos/*.yaml`) em tarefas: configuração × dobra externa (5) × dobra interna (4);
3. fixa o código no commit atual (as tarefas usam essa cópia, mesmo que você dê push depois);
4. submete o job array e, encadeado depois dele, o job de análise, que envia os resultados ao GitHub.

Como rodar: commit e push do código → no Apuana, `git pull` → abra este notebook (ou `python -m macdeg.cluster lancar ...`).
Rode primeiro com `MODO_TESTE = True` (minutos, números sem valor) e depois com `False`.

**Nunca escreva tokens neste notebook**: eles vêm do `.env` na raiz do projeto.

## 0. Configuração (edite aqui)

In [ ]:
EXPERIMENTO = "configs/experimentos/001_referencia.yaml"
MODO_TESTE = True          # True = passada rápida para validar o encadeamento; os números não servem
LOCAL = False              # True = roda aqui mesmo, sem SLURM (só para MODO_TESTE no seu computador)
ZIP = "~/macdeg_data/dataset_1024.zip"     # copiado para o disco local do nó em cada tarefa
DADOS = "~/macdeg_data/dataset_1024"       # pasta descompactada: só para conferir a assinatura aqui
SLURM = dict(partition="short-simple", qos="simple", max_paralelo=5, time="12:00:00", mem="32G", cpus=6)

## 1. Verificações

In [ ]:
import json, os, subprocess
from pathlib import Path
import yaml
from macdeg.ambiente import PROJECT, load_dotenv, which_secrets, push_results, commit_info
from macdeg.cluster import prepare, expand, dataset_signature, submit, run_local, task_dir
from macdeg.data.sources import resolve

os.chdir(PROJECT)
load_dotenv()
print("segredos presentes:", which_secrets())          # só True/False, nunca o valor
info = commit_info()
print("commit:", info["commit"][:10], "| mudanças sem commit:", info["codigo_alterado_sem_commit"])
if info["codigo_alterado_sem_commit"]:
    print("ATENÇÃO: o que não foi commitado NÃO vai para as tarefas.")

experimento = yaml.safe_load(open(EXPERIMENTO, encoding="utf-8"))
configs = expand(experimento)
print(f"{len(configs)} configuração(ões):")
for c in configs:
    print(" ", c["nome"], c["mudancas"] or "(base)")
root = resolve(DADOS)
print("assinatura dos dados:", dataset_signature(root, configs[0]["config"].get("splits", "splits_nested_seed0.csv")))

In [ ]:
# O envio ao GitHub é testado ANTES de qualquer treino (no mlp_fase3 ele falhou a fase inteira sem ninguém ver)
import tempfile, time
if not MODO_TESTE:
    with tempfile.TemporaryDirectory() as tmp:
        Path(tmp, "teste_envio.json").write_text(json.dumps({"quando": time.strftime("%Y-%m-%d %H:%M")}))
        ok = push_results(tmp, "resultados/_teste_envio", "teste de envio")
    assert ok, "O envio ao GitHub falhou: confira o GITHUB_TOKEN no .env antes de lançar."
    print("GitHub: envio testado e funcionando")

## 2. Lançar

In [ ]:
lote = prepare(EXPERIMENTO, DADOS, test=MODO_TESTE)
info_lote = json.loads((lote / "lote.json").read_text())
print(f"lote: {lote}\n{len(info_lote['tarefas'])} tarefas")
if LOCAL:
    run_local(lote, DADOS)
    from macdeg.analysis import analyze
    analyze(lote, DADOS, push=False)
else:
    job = submit(lote, ZIP, SLURM)
    print(f"job array {job} submetido; a análise roda sozinha quando ele terminar.")
    print(f"acompanhar: squeue -u $USER   |   logs: {lote}/logs/")

## 3. Depois: publicar o modelo final de uma configuração

Quando a análise apontar a configuração que vale publicar, rode abaixo com o nome dela (ex.: `referencia-00`).
São 5 treinos (um por dobra externa) e o ensemble deles vai, privado, para o Hugging Face.

In [ ]:
LOTE_ANALISADO = ""        # pasta do lote já analisado, ex.: ~/macdeg_runs/referencia-20261112-1030
PUBLICAR = []              # ex.: ["referencia-00"]
if LOTE_ANALISADO and PUBLICAR:
    final = prepare(None, DADOS, test=MODO_TESTE, final=PUBLICAR, from_batch=Path(LOTE_ANALISADO).expanduser())
    print("lote final:", final)
    if LOCAL:
        run_local(final, DADOS)
        from macdeg.analysis import analyze
        analyze(final, DADOS, push=False)
    else:
        print("job:", submit(final, ZIP, SLURM))